In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


DATA_DIR = Path("/kaggle/input/datasets/kartik2112/fraud-detection")       
TRAIN_FILE = DATA_DIR / "fraudTrain.csv"
TEST_FILE  = DATA_DIR / "fraudTest.csv"
SINGLE_FILE = DATA_DIR / "creditcard_transactions.csv"

OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TARGET = "is_fraud"

In [2]:
def read_csv_safely(path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path.resolve()}")
    df = pd.read_csv(path, low_memory=False)
    # Một số bản CSV lưu index thành cột Unnamed: 0
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed:")]
    return df

if TRAIN_FILE.exists() and TEST_FILE.exists():
    raw_train = read_csv_safely(TRAIN_FILE)
    raw_test = read_csv_safely(TEST_FILE)
    print("Dùng bộ train/test có sẵn.")
    print("Raw train:", raw_train.shape, "| Raw test:", raw_test.shape)
    source_mode = "pre_split"
else:
    raw_all = read_csv_safely(SINGLE_FILE)
    print("Dùng một file CSV duy nhất.")
    print("Raw data:", raw_all.shape)
    source_mode = "single_file"

if source_mode == "pre_split":
    df_train_source, df_test_source = raw_train.copy(), raw_test.copy()
else:
    df_train_source, df_test_source = raw_all.copy(), None

if TARGET not in df_train_source.columns:
    raise ValueError(f"Không tìm thấy cột target '{TARGET}'. Các cột hiện có: {list(df_train_source.columns)}")

Dùng bộ train/test có sẵn.
Raw train: (1296675, 22) | Raw test: (555719, 22)


In [3]:
def compute_historical_features(df):
    """
    Hàm tính các đặc trưng lịch sử cho từng thẻ (cc_num) theo đúng thứ tự thời gian.
    Đảm bảo KHÔNG nhìn thấy giao dịch hiện tại và tương lai (closed='left').
    """
    # 1. Đảm bảo dữ liệu được sắp xếp nghiêm ngặt theo thời gian và từng thẻ
    df = df.sort_values(["cc_num", "_sort_time"]).reset_index(drop=True)
    
    # 2. Khoảng cách thời gian (giây) từ giao dịch liền trước của cùng chủ thẻ
    df["time_since_last_trans_sec"] = (
        df.groupby("cc_num")["_sort_time"]
        .diff()
        .dt.total_seconds()
        .fillna(-1.0) # -1 biểu thị giao dịch đầu tiên được ghi nhận của thẻ
    )
    
    # 3. Tính Rolling Windows trên cột 'amt' theo thời gian thực (chỉ nhìn quá khứ với closed='left')
    # Đặt _sort_time làm index tạm thời để dùng time-based rolling của Pandas
    df_indexed = df[["cc_num", "_sort_time", "amt"]].set_index("_sort_time")
    grouped = df_indexed.groupby("cc_num")["amt"]
    
    # Đếm số lượng giao dịch trong 1h, 24h, 7 ngày trước đó
    df["trans_count_1h"]  = grouped.rolling("1h", closed="left").count().values
    df["trans_count_24h"] = grouped.rolling("24h", closed="left").count().values
    df["trans_count_7d"]  = grouped.rolling("7d", closed="left").count().values
    
    # Thống kê số tiền giao dịch trong 24h và 7 ngày trước đó
    df["sum_amt_24h"] = grouped.rolling("24h", closed="left").sum().values
    df["avg_amt_7d"]  = grouped.rolling("7d", closed="left").mean().values
    df["std_amt_7d"]  = grouped.rolling("7d", closed="left").std().values
    df["max_amt_7d"]  = grouped.rolling("7d", closed="left").max().values
    
    # Điền 0 cho các giao dịch đầu tiên chưa có lịch sử
    hist_cols = [
        "trans_count_1h", "trans_count_24h", "trans_count_7d",
        "sum_amt_24h", "avg_amt_7d", "std_amt_7d", "max_amt_7d"
    ]
    df[hist_cols] = df[hist_cols].fillna(0.0)
    
    # 4. Tính tỷ lệ đột biến chi tiêu (Ratio & Z-score xấp xỉ)
    # Cộng thêm 1.0 ở mẫu số để tránh chia cho 0 khi lịch sử bằng 0
    df["amt_to_avg_7d_ratio"] = df["amt"] / (df["avg_amt_7d"] + 1.0)
    df["amt_to_max_7d_ratio"] = df["amt"] / (df["max_amt_7d"] + 1.0)
    
    # Sắp xếp lại toàn bộ dataframe theo đúng thứ tự thời gian chung của hệ thống
    df = df.sort_values("_sort_time").reset_index(drop=True)
    return df


def prepare_features_with_history(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # 1. Xử lý cột thời gian giao dịch
    datetime_candidates = ["trans_date_trans_time", "transaction_time", "timestamp", "datetime"]
    dt_col = next((c for c in datetime_candidates if c in df.columns), None)
    if dt_col:
        dt = pd.to_datetime(df[dt_col], errors="coerce")
    elif "unix_time" in df.columns:
        dt = pd.to_datetime(df["unix_time"], unit="s", errors="coerce")
    else:
        raise ValueError("Không tìm thấy cột thời gian hợp lệ để tính lịch sử giao dịch!")

    df["transaction_hour"] = dt.dt.hour
    df["transaction_dayofweek"] = dt.dt.dayofweek
    df["transaction_month"] = dt.dt.month
    df["transaction_day"] = dt.dt.day
    df["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype("int8")
    df["_sort_time"] = dt

    # 2. Tính tuổi chủ thẻ (Customer Age) trước khi drop cột dob
    if "dob" in df.columns:
        dob_dt = pd.to_datetime(df["dob"], errors="coerce")
        df["customer_age"] = ((df["_sort_time"] - dob_dt).dt.days / 365.25).round(1)

    # 3. Khoảng cách Haversine giữa khách hàng và merchant
    coord_cols = {"lat", "long", "merch_lat", "merch_long"}
    if coord_cols.issubset(df.columns):
        lat1 = np.radians(pd.to_numeric(df["lat"], errors="coerce"))
        lon1 = np.radians(pd.to_numeric(df["long"], errors="coerce"))
        lat2 = np.radians(pd.to_numeric(df["merch_lat"], errors="coerce"))
        lon2 = np.radians(pd.to_numeric(df["merch_long"], errors="coerce"))
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        df["customer_merchant_distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    # 4. Tính toán Historical Features dựa trên cc_num và _sort_time
    if "cc_num" not in df.columns:
        raise ValueError("Cần cột 'cc_num' trong dữ liệu gốc để tính lịch sử giao dịch!")
    
    print("Đang tính toán Historical Features theo từng thẻ...")
    df = compute_historical_features(df)

    # 5. Xóa các cột định danh cá nhân (PII) và cc_num để chống model học vẹt
    drop_candidates = [
        "cc_num", "card_number", "first", "last", "name", "gender",
        "street", "city", "state", "zip", "dob", "job", "trans_num",
        "transaction_id", "id", "unix_time",
        "trans_date_trans_time", "transaction_time", "timestamp", "datetime",
        "lat", "long", "merch_lat", "merch_long"
    ]
    df = df.drop(columns=[c for c in drop_candidates if c in df.columns], errors="ignore")

    # 6. Chuẩn hóa cột Target
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df[df[TARGET].isin([0, 1])].copy()
    df[TARGET] = df[TARGET].astype("int8")

    return df

In [4]:
# 1. Định nghĩa lại hàm chia dữ liệu theo thứ tự thời gian
def chronological_split(df, train_ratio=0.70, val_ratio=0.15):
    df = df.sort_values("_sort_time", na_position="last").reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    train = df.iloc[:train_end].copy()
    val = df.iloc[train_end:val_end].copy()
    test = df.iloc[val_end:].copy()
    return train, val, test

# 2. Xử lý dữ liệu và chia tập Train / Validation / Test
if source_mode == "pre_split":
    # Đánh dấu nguồn để tách lại chính xác sau khi tính lịch sử nối tiếp
    df_train_source["_split_tag"] = "train_val"
    df_test_source["_split_tag"] = "test"
    
    combined_raw = pd.concat([df_train_source, df_test_source], ignore_index=True)
    combined_processed = prepare_features_with_history(combined_raw)
    
    # Tách lại tập train_val và test theo đúng tag ban đầu
    processed_train_source = combined_processed[combined_processed["_split_tag"] == "train_val"].drop(columns=["_split_tag"]).copy()
    test_df = combined_processed[combined_processed["_split_tag"] == "test"].drop(columns=["_split_tag"]).copy()
    
    # Chia train (85%) và validation (15%) theo đúng thứ tự thời gian
    train_df, val_df, _ = chronological_split(processed_train_source, train_ratio=0.85, val_ratio=0.15)
    test_df = test_df.sort_values("_sort_time", na_position="last").reset_index(drop=True)
else:
    processed_all = prepare_features_with_history(df_train_source)
    train_df, val_df, test_df = chronological_split(processed_all, TRAIN_RATIO, VAL_RATIO)

# 3. Bỏ cột _sort_time nội bộ trước khi lưu ra file CSV
for frame in (train_df, val_df, test_df):
    frame.drop(columns=["_sort_time"], inplace=True, errors="ignore")

print("Train:", train_df.shape, "| fraud rate:", round(train_df[TARGET].mean(), 5))
print("Validation:", val_df.shape, "| fraud rate:", round(val_df[TARGET].mean(), 5))
print("Test:", test_df.shape, "| fraud rate:", round(test_df[TARGET].mean(), 5))

Đang tính toán Historical Features theo từng thẻ...
Train: (1102173, 22) | fraud rate: 0.00578
Validation: (194502, 22) | fraud rate: 0.00583
Test: (555719, 22) | fraud rate: 0.00386


In [5]:
train_path = OUTPUT_DIR / "train.csv"
val_path = OUTPUT_DIR / "validation.csv"
test_path = OUTPUT_DIR / "test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Đã lưu:")
print(train_path.resolve())
print(val_path.resolve())
print(test_path.resolve())

Đã lưu:
/kaggle/working/processed_data/train.csv
/kaggle/working/processed_data/validation.csv
/kaggle/working/processed_data/test.csv


In [6]:
display(train_df.head())
display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "fraud_count": [
        int(train_df[TARGET].sum()),
        int(val_df[TARGET].sum()),
        int(test_df[TARGET].sum())
    ],
    "fraud_rate": [
        float(train_df[TARGET].mean()),
        float(val_df[TARGET].mean()),
        float(test_df[TARGET].mean())
    ]
}))

,merchant,category,amt,city_pop,is_fraud,transaction_hour,transaction_dayofweek,transaction_month,transaction_day,is_weekend,...,time_since_last_trans_sec,trans_count_1h,trans_count_24h,trans_count_7d,sum_amt_24h,avg_amt_7d,std_amt_7d,max_amt_7d,amt_to_avg_7d_ratio,amt_to_max_7d_ratio
0,"fraud_Rippin, Kub and Mann",misc_net,4.97,3495,0,0,1,1,1,0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.97,4.97
1,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,149,0,0,1,1,1,0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,107.23,107.23
2,fraud_Lind-Buckridge,entertainment,220.11,4154,0,0,1,1,1,0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,220.11,220.11
3,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,1939,0,0,1,1,1,0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.00,45.00
4,fraud_Keeling-Crist,misc_pos,41.96,99,0,0,1,1,1,0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.96,41.96


,split,rows,fraud_count,fraud_rate
0,train,1102173,6373,0.005782
1,validation,194502,1133,0.005825
2,test,555719,2145,0.003860


In [7]:
time_col = "trans_date_trans_time"

for name, df in [
    ("fraudTrain", raw_train),
    ("fraudTest", raw_test),
]:
    t = pd.to_datetime(df[time_col], errors="coerce")

    print(
        name,
        "| min:", t.min(),
        "| max:", t.max(),
        "| invalid:", t.isna().sum()
    )

fraudTrain | min: 2019-01-01 00:00:18 | max: 2020-06-21 12:13:37 | invalid: 0
fraudTest | min: 2020-06-21 12:14:25 | max: 2020-12-31 23:59:34 | invalid: 0
